# Lesson 02 - Exploring an Agent Framework

An agent framework gives you a small set of composable building blocks so you do not have to write the model-calling loop, the tool-dispatch code, or the conversation bookkeeping yourself. In LangChain and LangGraph those building blocks are:

- **Model client** – connects to an AI model endpoint and handles communication
- **Agent** – wraps a model client with instructions and tool definitions and runs the tool-calling loop
- **Tools** – extend agent capabilities with custom functions the model can call
- **Memory (thread)** – a checkpointer that keeps conversation history so multi-turn dialogue works

In this lesson, we'll build a **travel booking agent** that checks destination availability using these concepts.

## Setup

Prerequisites: run `pip install -r requirements.txt` in the repository root, copy `.env.example` to `.env`, fill in `LLM_BASE_URL`, `LLM_API_KEY`, `LLM_MODEL`, and run `python scripts/check_endpoint.py`.

The cell below loads those variables from `.env` and builds the chat model client. `ChatOpenAI` speaks the OpenAI Chat Completions protocol, so the same code works with DeepSeek, OpenAI, a local Ollama server, or any other compatible endpoint — only the three environment variables change. `LLM_EXTRA_BODY` carries optional provider-specific request options (for DeepSeek it turns thinking mode off).

In [1]:
import json
import os

from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(find_dotenv())

missing = [name for name in ("LLM_BASE_URL", "LLM_API_KEY", "LLM_MODEL") if not os.environ.get(name)]
if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Copy .env.example to .env in the repository root and fill them in."
    )

llm = ChatOpenAI(
    model=os.environ["LLM_MODEL"],
    base_url=os.environ["LLM_BASE_URL"],
    api_key=os.environ["LLM_API_KEY"],
    extra_body=json.loads(os.environ.get("LLM_EXTRA_BODY") or "null"),
)
print(f"Model client ready: {os.environ['LLM_MODEL']} @ {os.environ['LLM_BASE_URL']}")

Model client ready: deepseek-v4-pro @ https://api.deepseek.com/v1


## Understanding the Framework Architecture

The pieces fit together in layers:

```
ChatOpenAI  →  create_agent  →  tools
                             →  checkpointer (thread)
```

1. **Model client** – `ChatOpenAI` only talks HTTP. It formats a request for the Chat Completions API, sends it to `LLM_BASE_URL`, and parses the reply (including any tool calls the model asks for). It knows nothing about loops or conversations.
2. **Agent** – `create_agent` owns the **tool-calling loop**: it sends the conversation to the model, executes whichever tool the model requests, appends the result, and repeats until the model answers in plain text. The instructions you pass become the system prompt.
3. **Tools** – plain Python functions decorated with `@tool`. The agent hands the model a schema derived from the signature and docstring, and calls the function when the model asks.
4. **Memory (thread)** – a **checkpointer** saves the message history after every step. Each conversation is keyed by a `thread_id`; pass the same id and the agent remembers earlier turns, pass a new one and it starts fresh.

Let's build each layer step by step. The model client already exists from the setup cell above.

## Adding Tools with the @tool Decorator

Tools let agents take actions beyond generating text. The `@tool` decorator converts a regular Python function into something the agent can call.

Key points:
- The **docstring** becomes the tool description the model sees.
- The **type hints** define the argument schema.
- `Annotated[type, "description"]` adds a description for each parameter so the model understands what to pass.
- Tools run automatically when the model asks for them; a later lesson shows how to require human approval first.

In [2]:
from typing import Annotated

from langchain.tools import tool


@tool
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Creating an Agent with Tools

Now we combine the model client, instructions, and tools into an agent. The `system_prompt` defines the agent's persona and behaviour.

We also attach an `InMemorySaver` **checkpointer**. Without one the agent forgets everything between `invoke` calls; with one, every conversation identified by a `thread_id` is stored and replayed on the next turn.

`agent.invoke` returns the full message history — user message, the model's tool calls, the tool results, and the final reply. `reply_text` pulls the text out of the last message (some providers return content as a list of blocks rather than a single string).

In [3]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    llm,
    tools=[check_destination_availability],
    system_prompt=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    checkpointer=InMemorySaver(),
)


def reply_text(result) -> str:
    """Return the text of the last message; content can be a string or a list of blocks."""
    content = result["messages"][-1].content
    if isinstance(content, list):
        return "".join(block.get("text", "") for block in content if isinstance(block, dict))
    return content

## Multi-Turn Conversations with a Thread

A **thread** is a conversation the checkpointer remembers. You select it with a config dictionary, `{"configurable": {"thread_id": "..."}}`, passed as the second argument to `invoke`. By passing the same config to each call, the agent has access to the full conversation history and can refer back to earlier messages.

The agent can call `check_destination_availability` during any turn — the tool-calling loop runs inside every `invoke`.

The tool answers only for a destination you *name*; adding a `list_destinations` tool is a good exercise once you finish the notebook.

In [4]:
session = {"configurable": {"thread_id": "travel-availability-1"}}

# Turn 1: Ask about specific destinations
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Which of Barcelona, Tokyo, Cape Town, Vancouver and Dubai are available?"}]},
    session,
)
print(f"Agent: {reply_text(result)}")

# Turn 2: Follow-up question — the agent remembers the conversation
result = agent.invoke(
    {"messages": [{"role": "user", "content": "I'd like to go somewhere warm. What's available?"}]},
    session,
)
print(f"\nAgent: {reply_text(result)}")

Agent: Here are the results:

**Available for booking:**
- ✅ Barcelona
- ✅ Tokyo
- ✅ Vancouver

**Not available for booking:**
- ❌ Cape Town
- ❌ Dubai

Would you like any recommendations or help planning a trip to one of the available destinations?

Agent: Based on your preference for warm weather, here's how the available destinations stack up:

**Available warm destinations:**
- ✅ **Barcelona** – Mediterranean climate with warm, sunny weather, especially a great choice for beach and culture.
- ✅ **Vancouver** – typically mild, though it's better known for a temperate (rather than hot) climate; warmth depends on the season.

**Tokyo** is also available but tends to have more distinct seasons, with hot and humid summers but cooler winters.

**Cape Town and Dubai** (both unavailable) would have been strong warm-weather options, but they're not currently bookable.

If you're looking for the warmest option right now, **Barcelona** would likely be your best bet. Would you like me to help y

## A Second Tool and Streaming

Tools can hold state of their own. The tool below picks a random vacation destination and remembers the previous pick so that two calls in a row never return the same place — handy when the user rejects a suggestion and asks for another.

We create a second agent around it, again with a checkpointer, and this time **stream** the replies. `agent.astream(..., stream_mode="messages")` yields `(token, metadata)` pairs for every message chunk the graph produces; we print the chunks that come from the model node and start a new line whenever the tools node runs, so the text before and after a tool call do not run together. Jupyter supports top-level `await`, so the `async for` below runs as-is.

Both turns share one `thread_id`, so on the second turn the agent knows which destination was rejected.

In [5]:
import random

# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


random_trip_agent = create_agent(
    llm,
    tools=[get_random_destination],
    system_prompt="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    checkpointer=InMemorySaver(),
)

user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]
session = {"configurable": {"thread_id": "random-trip-1"}}

for turn, user_input in enumerate(user_inputs, start=1):
    print(f"--- turn {turn} ---")
    print(f"User: {user_input}\nAgent: ", end="")
    async for token, metadata in random_trip_agent.astream(
        {"messages": [{"role": "user", "content": user_input}]}, session, stream_mode="messages"
    ):
        if metadata.get("langgraph_node") == "model" and getattr(token, "content", None):
            print(token.content, end="", flush=True)
        elif metadata.get("langgraph_node") == "tools":
            print()  # blank line between the text before and after a tool call
    print("\n")

--- turn 1 ---
User: Plan me a day trip.
Agent: I'll find a random destination for your day trip!
# 🌍 Day Trip: Cape Town, South Africa

Here's a packed (but relaxed) one-day itinerary for Cape Town:

## 🏔️ Morning — Iconic Landmarks
- **Sunrise at Table Mountain** – Take the Aerial Cableway up for panoramic views of the city, ocean, and Table Bay. (Go early to beat the crowds and the clouds.)
- **Breakfast** in the V&A Waterfront area – grab a coffee and pastry with harbor views.

## 🐧 Midday — Coastal & Culture
- **Bo-Kaap neighborhood** – a quick stroll through the colorful houses, perfect for photos.
- **Chapman's Peak Drive** – one of the world's most scenic coastal drives (pull over for viewpoints).
- **Lunch** at **Hout Bay** – try fresh fish & chips at the harbor market.

## 🐧 Afternoon — Wildlife & Nature
- **Boulders Beach** – see the famous African penguin colony wandering the beach.
- Optional: visit **Cape Point** & the Cape of Good Hope for dramatic cliffs and coastline.


## Summary

In this lesson you explored the four building blocks of an agent framework:

| Concept | What You Learned |
|---------|------------------|
| **Model client** | `ChatOpenAI` connects to any OpenAI-compatible endpoint; the provider is just configuration |
| **Agent** | `create_agent` bundles a model client with instructions and tools and runs the tool-calling loop |
| **Tools** | The `@tool` decorator exposes Python functions (docstring + type hints) for the agent to call |
| **Memory** | `InMemorySaver` + a `thread_id` maintain conversation history across multiple turns |

These building blocks compose together to create agents that can hold natural conversations, call external functions, and maintain context — the foundation for more advanced agentic patterns in later lessons.